# A 2-D U-Net forward pass

Build a three-class network and inspect its probability maps. No downloads or
training are involved. These random weights do not produce anatomical segmentation.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

## Construct a small network and a reproducible input

In [ ]:
torch.manual_seed(42)
model = antstorch.create_unet_model_2d(
    input_channel_size=1, number_of_outputs=3,
    number_of_filters=(8, 16, 32), mode="classification",
).to(device).eval()
image = torch.rand(1, 1, 64, 64, device=device)
with torch.inference_mode():
    probabilities = model(image)
print("Output layout (batch, classes, height, width):", tuple(probabilities.shape))

## Check the output contract
Classification mode returns a probability per class and voxel. The class
probabilities should be finite, lie in [0, 1], and sum to one.

In [ ]:
assert probabilities.shape == (1, 3, 64, 64)
assert torch.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
torch.testing.assert_close(probabilities.sum(dim=1), torch.ones_like(image[:, 0]))
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
axes[0].imshow(image[0, 0].cpu(), cmap="gray")
axes[0].set_title("Random input")
for index in range(3):
    axes[index + 1].imshow(probabilities[0, index].cpu(), vmin=0, vmax=1)
    axes[index + 1].set_title(f"Class {index} probability")
for ax in axes: ax.axis("off")
plt.show()

Train a model on suitable labeled images before interpreting its predictions.